# 7.3 Clusters in graphs — and whether they are there

[07.2](07.2-graph-properties.ipynb) measured one node at a time. This notebook asks the group
question: **does this network fall apart into groups?**

You do not have to find the groups by hand. There are algorithms that take a graph and return
its communities, and when the groups are there they find them: on the karate club below they
recover a split that really happened, the same split your eye already suspects when it looks
at the drawing. That is the first half of the notebook — clustering as a confirmation of what
you can see, from a method that did not look at the answer.

The second half is the catch. Every clustering algorithm returns clusters, including on a
graph with no groups in it whatsoever; there are no error bars and nothing refuses. So the
technique is **cluster the graph, then check that the clusters are not what any graph of that
shape would have given you** — and know which shapes fool the check.

The worked example is Zachary's karate club, for one reason: the true answer was recorded
independently of the data. Barabási's [*Network Science*](http://networksciencebook.com/) chapter
[9](http://networksciencebook.com/chapter/9) is the reference — modularity and its limits in 9.4,
testing communities in 9.6, and the Louvain algorithm in advanced topic 9.C.

In [ ]:
import itertools

import networkx as nx
import numpy as np
import pandas as pd
from goad_toolkit.visualizer import HeatmapPlot, LinePlot, PlotSettings
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

from scripts.graphs import (
    GraphPlot,
    MentionEdges,
    best_partition,
    giant_component,
    labels,
    modularity_check,
    to_graph,
)
from scripts.pipelines import BuildTimestamp, build_irc_pipeline
from wa_analyzer.data import load_showcase
from wa_analyzer.network_analysis import Config, GraphBuilder

## 7.3.1 A split that actually happened

Wayne Zachary watched a university karate club for three years in the 1970s and recorded who
spent time with whom outside the dojo. During the study the administrator ("Mr. Hi") and the
instructor fell out, and the club split in two. Zachary wrote down which half each of the 34
members joined.

So this graph comes with a label column that was *not* derived from the edges. That is rare and
it is why the dataset is famous: it lets you ask whether an algorithm that never saw the labels
recovers them.

In [ ]:
karate = nx.karate_club_graph()
truth = np.array([0 if karate.nodes[n]["club"] == "Mr. Hi" else 1 for n in karate.nodes()])
pos = nx.spring_layout(karate, seed=46)

club = PlotSettings(figsize=(11, 5), max_cols=2, title="Zachary's karate club, 34 members",
                    subplot_titles=["the graph", "the split that happened"])
host = GraphPlot(club)
fig, axes = host.create_figure(n_plots=2)
host.plot_on_axes(GraphPlot(club), axes[0], data=karate, pos=pos, node_size=180, with_labels=True)
_ = host.plot_on_axes(GraphPlot(club), axes[1], data=karate, pos=pos, node_size=180,
                      with_labels=True, color=["#4c72b0" if t == 0 else "#c44e52" for t in truth])
print(f"{karate.number_of_nodes()} members, {karate.number_of_edges()} ties, "
      f"{int((truth == 0).sum())} went with Mr. Hi and {int((truth == 1).sum())} with the officer")

## 7.3.2 The Laplacian, and why its *smallest* eigenvalues

Two matrices describe a graph completely (Barabási 2.4). The **adjacency matrix** $A$ has
$A[i,j] = 1$ when $i$ and $j$ are connected. The **degree matrix** $D$ is diagonal, holding each
node's degree. The **Laplacian** is their difference:

$$L = D - A$$

Every row of $L$ sums to zero — your degree on the diagonal, minus one for each neighbour. That
is not a coincidence, it is the reason the whole method works: it means the vector of all ones is
an eigenvector with eigenvalue 0, always, for every graph.

In PCA (06.1) the *largest* eigenvalues mattered, because they carried the most variance. Here it
is the smallest, and the reason is worth holding on to. For any vector $x$ over the nodes,

$$x^{\top} L x = \sum_{(i,j) \in E} (x_i - x_j)^2$$

— the total disagreement between connected nodes. An eigenvector with a *small* eigenvalue is a
way of assigning numbers to people that neighbours mostly agree on. The smallest is the trivial
one (give everybody the same number, disagreement zero). The **second smallest** is the smallest
non-trivial one: the way to split the numbers so that as few edges as possible are cut. It has a
name, the **Fiedler vector**, and its eigenvalue measures how easy the graph is to cut in two.

The count of zero eigenvalues is the number of connected components — a disconnected graph can
assign a different constant to each piece for free.

In [ ]:
A = nx.to_numpy_array(karate)
D = np.diag(A.sum(axis=1))
L = D - A

values, vectors = np.linalg.eigh(L)
order = np.argsort(values)
values, vectors = values[order], vectors[:, order]
print(f"L is {L.shape}, rows sum to {L.sum(axis=1).max():.1e}")
print(f"the ten smallest eigenvalues: {np.round(values[:10], 3)}")
print(f"eigenvalues equal to zero: {int(np.isclose(values, 0, atol=1e-9).sum())} "
      f"-> {nx.number_connected_components(karate)} connected component")
print(f"the gap after the Fiedler value: {values[2] / values[1]:.2f}x")

spectrum = PlotSettings(figsize=(7, 4), title="Karate club: the Laplacian spectrum",
                        xlabel="eigenvalue index", ylabel="eigenvalue")
fig, ax = LinePlot(spectrum).plot(data=pd.DataFrame({"i": range(len(values)), "v": values}),
                                  x="i", y="v", marker="o", color="#666666")
_ = ax.axvline(1, color="#c44e52", linestyle="--")

One zero, then 1.19, then 2.39: the second eigenvalue is a factor two below the third. **That gap
is the only thing in this notebook that tells you how many clusters to look for**, and here it
says two. Hold on to how rare that will turn out to be.

## 7.3.3 The Fiedler vector, coloured by sign

The second eigenvector gives every member a number. Colour by whether that number is above or
below zero and compare against the split Zachary recorded.

In [ ]:
fiedler = vectors[:, 1]
side = (fiedler > 0).astype(int)
agreement = max((side == truth).mean(), 1 - (side == truth).mean())
wrong = [n for n, s, t in zip(karate.nodes(), side, truth)
         if (s == t) != ((side == truth).mean() > 0.5)]
print(f"Fiedler split: {int(side.sum())} vs {int((1 - side).sum())} members")
print(f"agreement with the recorded split: {agreement:.1%}  (ARI {adjusted_rand_score(truth, side):.3f})")
print(f"members on the wrong side: {wrong}")

split = PlotSettings(figsize=(11, 5), max_cols=2, title="The second eigenvector against the record",
                     subplot_titles=["sign of the Fiedler vector", "the split that happened"])
host = GraphPlot(split)
fig, axes = host.create_figure(n_plots=2)
host.plot_on_axes(GraphPlot(split), axes[0], data=karate, pos=pos, node_size=180, with_labels=True,
                  color=["#4c72b0" if s == 0 else "#c44e52" for s in side])
_ = host.plot_on_axes(GraphPlot(split), axes[1], data=karate, pos=pos, node_size=180,
                      with_labels=True, color=["#4c72b0" if t == 0 else "#c44e52" for t in truth])

**Thirty-three of thirty-four.** An eigenvector of a matrix built from nothing but who trained
with whom reproduces a social rupture that happened two years later, and misplaces one person.

That person is member 8, and the miss is the interesting part rather than a rounding error.
Member 8 has ties on both sides; Zachary's own account records that he joined the officer's club
for a reason outside the graph — he was three weeks from a black belt under Mr. Hi and would have
lost the rank by switching. **The one node the mathematics gets wrong is the one whose decision
was made on information the network does not contain.** That is the honest shape of a good
result, and it is worth remembering when you have no labels to be wrong against.

Look back at the drawing in §7.3.1 and the split is not a surprise: there is a left side around
member 0 and a right side around member 33, and your eye found them before any eigenvector did.
The algorithm's contribution is that it found them *without* the eye, from the edges alone — a
confirmation, which is worth most when the drawing is a hairball. The adjacency matrix shows the
same thing in a form that does not depend on a layout, as long as you choose the order of its
rows: shuffled it is noise, sorted by the side of the Fiedler vector it is two blocks.

In [ ]:
def adjacency(graph, order):
    return pd.DataFrame(nx.to_numpy_array(graph, nodelist=order, weight=None), index=order, columns=order)


shuffled = list(np.random.default_rng(0).permutation(list(karate.nodes())))
sides = sorted(karate.nodes(), key=lambda n: (side[n], -karate.degree(n)))
blocks = PlotSettings(figsize=(11, 5), max_cols=2, title="The karate club as a matrix",
                      subplot_titles=["members in random order", "sorted by side of the Fiedler vector"],
                      xlabel="", ylabel="")
host = HeatmapPlot(blocks)
fig, axes = host.create_figure(n_plots=2)
for ax, order in zip(axes, [shuffled, sides]):
    host.plot_on_axes(HeatmapPlot(blocks), ax, data=adjacency(karate, order), cmap="Greys",
                      annot=False, cbar=False, xticklabels=False, yticklabels=False)

## 7.3.4 More clusters, and the price of asking for them

The sign of one eigenvector splits a graph in two. For $k$ clusters the standard move is to hand
the first few eigenvectors to KMeans and treat each node's eigenvector values as its coordinates
— the graph embedded in a space where connected nodes sit close (06.1's move, on a different
matrix).

In [ ]:
rows = []
for k in (2, 3, 4, 5):
    embedding = vectors[:, 1:k]
    assignment = KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(embedding)
    rows.append({"k": k, "eigenvectors used": f"2..{k}",
                 "sizes": np.bincount(assignment).tolist(),
                 "ARI vs the record": adjusted_rand_score(truth, assignment)})
print(pd.DataFrame(rows).round(3).to_string(index=False))

At $k=2$ KMeans on the Fiedler vector alone reproduces the sign split exactly, ARI 0.88. Every
larger $k$ is worse — 0.68, 0.56, 0.59 — and at $k=4$ and $k=5$ one of the "communities" is a
single member, which is what an algorithm asked for four groups does when the graph has two.

Nothing in the output says which row to believe. The eigenvalue plot did, once: the gap after
the second eigenvalue said two, and two is the row that works. **The number of clusters is not a
parameter you tune until the picture looks good** — it is a claim, and the spectrum is the only
evidence for it that the graph itself provides. When there is no gap, there is no evidence.

## 7.3.5 Modularity, and why it is not a score

Louvain (Barabási 9.12) takes a different route: it looks for a partition that maximises
**modularity** $Q$ — how many more edges fall inside communities than would if the same nodes
with the same degrees were wired at random (Barabási 9.4).

In [ ]:
partition = best_partition(karate)
karate_labels = labels(karate, partition)
print(f"louvain: Q = {nx.community.modularity(karate, partition, weight=None):.3f}, "
      f"{len(partition)} communities, sizes {sorted((len(c) for c in partition), reverse=True)}")
print(f"ARI against the recorded split: "
      f"{adjusted_rand_score(truth, karate_labels.reindex(karate.nodes())):.3f}")

coarse = best_partition(karate, resolution=0.6)
coarse_labels = labels(karate, coarse)
print(f"\nlouvain at resolution 0.6: {len(coarse)} communities, "
      f"sizes {sorted((len(c) for c in coarse), reverse=True)}")
print(f"ARI against the recorded split: "
      f"{adjusted_rand_score(truth, coarse_labels.reindex(karate.nodes())):.3f}")

Left alone, louvain splits the club into **four** communities, not two, and its agreement with
the recorded split is 0.47 — it has cut both real halves in half. This is modularity's
**resolution limit** (Barabási 9.4, "Limits of Modularity"): maximising $Q$ has a built-in
preferred community size that depends on the size of the whole graph, so it will merge genuinely
separate small groups in a large network and subdivide large ones in a small network. Turn the
resolution down to 0.6 and you get two communities of seventeen and ARI 0.77 — most of the right
answer, from a parameter chosen because we already knew the right answer.

**That is the trap this whole notebook is about.** The four-community answer and the
two-community answer are both louvain output, both have a defensible $Q$, and only one of them
matches reality. Without the label column there is nothing in the run that prefers either.

## 7.3.6 The number that makes modularity mean something

$Q$ has no natural zero. A graph wired at random still has some partition that scores well,
because in any finite random graph some regions happen to be denser than others — and the
sparser the graph, the more room there is for that accident. So $Q = 0.4$ is not a result. The
result is $Q$ **compared against the same measurement on a graph that has the same degrees and
nothing else** (Barabási 9.6).

`modularity_check` builds that comparison: twenty copies of the graph with the edges rewired by
repeated double swaps, which keeps every node's degree exactly, clusters each one the same way,
and reports how many standard deviations the real graph sits above them.

In [ ]:
pipeline = build_irc_pipeline()
pipeline.add(BuildTimestamp)
irc = pipeline.apply(load_showcase("ubuntu_irc"))
chat = irc[(irc.channel == "#ubuntu-uk") & (irc.date.dt.year == 2015)].copy()

mentions = giant_component(to_graph(MentionEdges()(chat), nodes=chat.author.unique()))
builder = GraphBuilder(Config(time_col="timestamp", node_col="author", seconds=600, datafile=None))
nearby = giant_component(builder.build(chat, edge_seconds=600))

checks = {
    "karate club": modularity_check(karate),
    "florentine families": modularity_check(nx.florentine_families_graph()),
    "irc mentions": modularity_check(mentions),
    "irc nearby, 600s": modularity_check(nearby),
}
sizes = {"karate club": karate, "florentine families": nx.florentine_families_graph(),
         "irc mentions": mentions, "irc nearby, 600s": nearby}
table = pd.DataFrame([
    {"graph": name, "nodes": sizes[name].number_of_nodes(), "edges": sizes[name].number_of_edges(),
     "Q": check.q, "Q rewired": check.null_mean, "spread": check.null_std,
     "z": check.z, "k": check.k}
    for name, check in checks.items()
])
print(table.round(3).to_string(index=False))

Read the `Q` column and the `z` column against each other, because they rank the four graphs in
almost opposite orders.

**The Florentine families score $Q = 0.40$ — and $z = +0.6$.** Rewire those twenty marriages at
random, keeping each family's number of in-laws, and you get a modularity just as high. The
fifteen-node graph that [07.2](07.2-graph-properties.ipynb) read so confidently for centrality
has no community structure that survives the check. It is too small for the question: with twenty
edges, *any* partition into four groups leaves a lot of edges inside by luck.

**The karate club scores $Q = 0.42$ — barely higher — and $z = +11$.** Almost the same
modularity, an entirely different verdict. This is the graph with real communities in it.

**The IRC mention graph scores $Q = 0.22$ and $z = +2.6$.** Two and a half standard deviations is
not nothing, and it is also not much: the mention network has slightly more structure than a
random network with the same degree sequence, and slightly is the honest word. Almost all of its
modularity is the modularity any graph of that shape would have had.

The lesson generalises past graphs. **A statistic with no natural zero needs a null, and the null
has to hold constant everything you are not asking about** — here, every node's degree, because
a heavy-tailed degree sequence produces high modularity all by itself. Reporting $Q$ alone is
like reporting an $R^2$ with no idea how many predictors were fitted.

## 7.3.7 Triangles make communities too

The rewired null holds every node's degree fixed and scrambles the rest — including the
triangles. [07.2](07.2-graph-properties.ipynb) met triangles as triadic closure: friends of
friends become friends. That is a local habit, one person introducing two of their friends to
each other, and it has nothing to do with groups. Does it fool the check?

Build a network with no groups at all — 100 people wired at random, most of them with one or two
ties — and then let every person introduce each pair of their friends to each other with
probability 0.8. Nobody chooses a group; everybody just introduces.

In [ ]:
def introduce(graph, p, seed):
    """Every person introduces each pair of their friends to each other with probability p."""
    rng = np.random.default_rng(seed)
    closed = graph.copy()
    for person in graph.nodes():
        for a, b in itertools.combinations(sorted(graph.neighbors(person)), 2):
            if rng.random() < p:
                closed.add_edge(a, b)
    return closed


degrees = np.random.default_rng(0).geometric(0.5, size=100)
degrees[0] += degrees.sum() % 2
strangers = nx.Graph(nx.configuration_model(degrees, seed=0))
strangers.remove_edges_from(nx.selfloop_edges(strangers))
strangers = giant_component(strangers)
friends = introduce(strangers, p=0.8, seed=0)

for label, graph in [("random, no groups", strangers), ("after introductions", friends)]:
    print(f"{label:20s} {graph.number_of_edges():3d} edges, clustering {nx.average_clustering(graph):.2f}  "
          f"{modularity_check(graph)}")

In [ ]:
pos = nx.spring_layout(friends, seed=2)
palette = ["#4c72b0", "#c44e52", "#55a868", "#8172b2", "#ccb974", "#64b5cd", "#937860", "#da8bc3",
           "#8c8c8c", "#e5ae38"]
panels = PlotSettings(figsize=(11, 5), max_cols=2, title="Louvain on a network without groups",
                      subplot_titles=["random, no groups", "after introductions"])
host = GraphPlot(panels)
fig, axes = host.create_figure(n_plots=2)
for ax, graph in zip(axes, [strangers, friends]):
    groups = labels(graph, best_partition(graph))
    host.plot_on_axes(GraphPlot(panels), ax, data=graph, pos=pos, node_size=60, edge_alpha=0.4,
                      color=[palette[groups[n] % len(palette)] for n in graph.nodes()])

**The random network passes the check as it should:** $Q = 0.70$, a higher modularity than the
karate club, and $z = +1$ — rewired copies score just as high, so there is nothing to report.
That is §7.3.6's lesson once more: a sparse random graph cuts into tidy pieces, and only the
null says the pieces mean nothing.

**After the introductions the same people score $z = +26$**, more than twice the karate club's
$+11$. The check now calls these the most convincing communities in the notebook. The pictures
are no help either way: louvain colours both networks into tidy clumps, because it colours
every network into tidy clumps. There are no groups in this network. Every clump is
one person's circle of friends, closed into a dense knot by introductions, and the rewired null
cannot know that because rewiring destroys the triangles along with everything else.

This is not a curiosity of a toy network. Peixoto (2022, *Physical Review X* 12, 011004) built a
model that tells community edges and triadic-closure edges apart, and ran it on a friendship
network of American high-school students from the same Add Health survey as 07.2's suicide
study. The standard method finds 26 groups. Once triangles are allowed to be triangles, 9 are
left, and they line up with the school grades; one of the 26, a clump of 8th-graders, turned out
to be nothing but the friends of one student having become friends with each other.

So a strong $z$ says the graph is not random. It does not say *why*. **Groups are one way to be
not random; a habit of introducing your friends is another**, and on a chat graph — where two
people who reply to the same person in the same minute end up connected — the second is built
into the edge rule.

## 7.3.8 The chat graph: stable, and visible?

Now the IRC mention graph from §7.3.6, the one with $z = +2.6$. Two checks that need no null at
all. First the matrix, sorted by louvain's communities, next to the karate club's. Then the same
clustering on two halves of the year.

In [ ]:
def by_community(graph, partition):
    """Nodes grouped by community, largest first, and by degree within each community."""
    groups = labels(graph, partition)
    return sorted(graph.nodes(), key=lambda n: (groups[n], -graph.degree(n)))


nodes = list(mentions.nodes())
compare = PlotSettings(figsize=(11, 5), max_cols=2, title="Adjacency matrices sorted by louvain community",
                       subplot_titles=["karate club, 34 members", "irc mention graph, 256 people"],
                       xlabel="", ylabel="")
host = HeatmapPlot(compare)
fig, axes = host.create_figure(n_plots=2)
for ax, graph, partition in [(axes[0], karate, best_partition(karate, resolution=0.6)),
                             (axes[1], mentions, best_partition(mentions, weight="weight"))]:
    order = by_community(graph, partition)
    host.plot_on_axes(HeatmapPlot(compare), ax, data=(adjacency(graph, order) > 0).astype(int), cmap="Greys",
                      annot=False, cbar=False, xticklabels=False, yticklabels=False)
print(f"irc communities: {sorted((len(c) for c in best_partition(mentions, weight='weight')), reverse=True)}")

Both matrices are built around hubs: in the karate club they are member 0 and member 33, the
dark first row and column of each block. The difference is where the hub rows stop. In the
karate club they stop at the edge of their own block, and inside the block the other members are
connected among themselves too: two squares. In the chat each community is a hub and the people
who address it — a dark corner with a line along the top and the left, and empty space where the
members would have talked to each other — and the hub rows run on into the other communities.
That is a network of people who all talk *to the same few people*, and a community algorithm has
to cut it somewhere; the cuts are where the hubs' circles happen to overlap least.

So the second check: if the communities are a property of these people, they should come back
when the same rule runs on a different stretch of time.

In [ ]:
seed_labels = [labels(mentions, nx.community.louvain_communities(mentions, weight="weight", seed=s))
               .reindex(nodes).to_numpy() for s in range(10)]
seed_ari = [adjusted_rand_score(a, b) for a, b in itertools.combinations(seed_labels, 2)]
print(f"across 10 louvain seeds:      ARI {np.mean(seed_ari):.3f} +- {np.std(seed_ari):.3f}")

base = labels(mentions, best_partition(mentions, weight="weight")).reindex(nodes).to_numpy()
for resolution in (0.75, 1.25, 1.5):
    other = labels(mentions, best_partition(mentions, weight="weight", resolution=resolution))
    print(f"resolution {resolution} vs 1.0:      "
          f"ARI {adjusted_rand_score(base, other.reindex(nodes)):.3f}")

In [ ]:
half = pd.Timestamp("2015-07-01")
first = giant_component(to_graph(MentionEdges()(chat[chat.timestamp < half])))
second = giant_component(to_graph(MentionEdges()(chat[chat.timestamp >= half])))
first_labels = labels(first, best_partition(first, weight="weight"))
second_labels = labels(second, best_partition(second, weight="weight"))
survivors = sorted(set(first.nodes()) & set(second.nodes()))
print(f"first half: {first.number_of_nodes()} people, second half: {second.number_of_nodes()}, "
      f"in both: {len(survivors)}")
print(f"January-June vs July-December:  ARI "
      f"{adjusted_rand_score(first_labels.reindex(survivors), second_labels.reindex(survivors)):.3f}")

nearby_labels = labels(nearby, best_partition(nearby, weight="weight"))
shared = sorted(set(nearby.nodes()) & set(mentions.nodes()))
print(f"mention rule vs nearby rule:    ARI "
      f"{adjusted_rand_score(labels(mentions, best_partition(mentions, weight='weight')).reindex(shared), nearby_labels.reindex(shared)):.3f}")

The seeds agree with each other well enough — louvain is randomised but it is not chaotic. Every
other line is a failure.

**The same rule on the same channel, six months apart, agrees at ARI 0.08.** Of the eighty people
who were around in both halves of 2015, knowing who clustered together in January tells you
essentially nothing about who clusters together in August — 0.08 is a hair above the zero that
means "no better than shuffling the labels". Whatever the algorithm found, it is not a property
of these people that persists for six months.

**The mention rule and the nearby rule agree at ARI 0.18** on the 255 people both graphs contain.
07.1 asked whether the choice of rule changes who looks busiest, and the answer was mostly no: the
two rules rank people's *degrees* alike, spearman 0.90. Their *communities* have almost nothing in
common. ARI is to a clustering what spearman is to a ranking, the same check of whether a result
survives a choice, and the two checks disagree. Agreement about who is busy is not agreement about
who belongs together.

Turning the resolution knob is a third failure of the same kind: the clusters do not get finer,
they get *rearranged*.

## 7.3.9 The verdict

Laid out plainly, because a null result is only worth anything if it is stated as sharply as a
positive one would have been:

| graph | $Q$ | $z$ vs rewired | what the matrix shows | what is behind the clusters |
|---|---|---|---|---|
| karate club | 0.42 | +11 | two squares | the split that happened |
| florentine families | 0.40 | +0.6 | — | nothing the null cannot also produce |
| random graph | 0.70 | +1 | — | nothing |
| random graph + introductions | 0.61 | +26 | clumps | one person's friends each, no groups |
| irc mentions | 0.22 | +2.6 | hubs, no squares | nothing stable: ARI 0.08 between halves |

Read the $Q$ column on its own and the random graph wins. Read the $z$ column on its own and the
introductions win. Only the karate club has groups, and it is the one row where something outside
the graph confirms it.

**The `#ubuntu-uk` mention network does not have defensible communities.** Its modularity is
barely above a graph with the same degrees wired at random; its matrix shows hubs, not squares; its
clusters do not survive splitting the year in two or changing the edge rule.

What you may say: *"Louvain finds nine communities in the mention graph at $Q = 0.22$, 2.6
standard deviations above twenty rewirings that keep every degree. The partition does not
replicate between the first and second half of 2015 (ARI 0.08) and does not agree with the
nearby rule (ARI 0.18). I do not think these are social groups."*

What you may not say: any sentence naming a cluster.

## 7.3.10 Your turn

Your own chat is small, so louvain will return communities immediately and they will look
convincing. Run the null before you look at them.

```python
from scripts.graphs import best_partition, giant_component, labels, modularity_check
from wa_analyzer.data import load_own_chat
from wa_analyzer.network_analysis import Config, GraphBuilder

own = load_own_chat()
own["timestamp"] = pd.to_datetime(own["timestamp"])
graph = giant_component(GraphBuilder(Config(time_col="timestamp", node_col="author",
                                            seconds=600, datafile=None)).build(own, edge_seconds=600))
print(modularity_check(graph))            # Q, the rewired Q, and z

first = own[own.timestamp < own.timestamp.median()]
second = own[own.timestamp >= own.timestamp.median()]
# ...build a graph from each, cluster both, and compare with adjusted_rand_score
```

> **Your turn.**
>
> 1. Look at your graph before clustering it. Do you see groups by eye? Write down which, so the
>    algorithm can confirm or contradict you rather than inform you.
> 2. Run `modularity_check` on your own chat graph. Report $Q$, the rewired $Q$ and $z$ together
>    — never $Q$ alone. In a group of eight the graph is nearly complete and $z$ will be small;
>    say so.
> 3. Draw the adjacency matrix sorted by the clusters, next to the graph. Squares, hubs, or
>    neither?
> 4. Split your chat in half by time, cluster each half, and report the ARI between them on the
>    people present in both. This is the single most convincing number you can produce about a
>    clustering, in either direction.
> 5. If your chat has a structure you *know* about — a family half and a work half — use it as
>    karate-club labels and report the ARI. You are one of very few people in this course with a
>    real label column; use it.

## 7.3.11 What to write down

1. **What you saw by eye first**, and whether the clustering agreed.
2. **$Q$ and the null together**, always, plus how the null was built — and that a rewired null
   does not rule out triangles.
3. **How many clusters, and why that number.** An eigengap, a resolution you can justify, or an
   admission that you picked it.
4. **One stability number** — seeds, threshold, or a split of the data, ideally the last.
5. **The sentence you are entitled to.** Write it out and check every clause against a number
   above it.

---

**Where this goes next.** Nothing, in this course — [07.1](07.1-social_graphs.ipynb) said the
tools stop at lesson 7. What does not stop is the shape of all three notebooks in this lesson,
which is the shape of the whole course: a technique that runs, a decision inside it that the
technique cannot make, and a check that tells you whether the output survived the decision. The
edge rule in 07.1, the layout and the choice of measure in 07.2, the number of clusters and the
null model here.

You will be asked to defend one of those choices, and "a null result, bounded" is a defence. "The
algorithm returned nine communities" is not.